# Prediction-interval calibration

Scorecard dimension D9 (uncertainty calibration). This page measures whether
a stated 90% prediction interval contains 90% of new responses, and how wide
the interval is compared with the narrowest interval the known generating
process allows.

:::{note} Recorded before the response-units change
The mboostrs CQR (conformalized quantile regression) and discovery results on this page are recorded results. They were recorded with mboostrs 0.2.3 or earlier, when distributional (LSS) and `Quantile` fits boosted the response in the units it was given. Since then these fits divide the response by its standard deviation, boost the rescaled response and map the fitted model back (`response_scale="auto"`), so a step has the same size in any units; `Quantile` also starts at the `tau`-quantile of the response, and takes steps of 0.3 when no control is given. Refitting on the current version gives different numbers. `response_scale=None` with the settings the page states reproduces the recorded fits; [Response units](../implementation/parity_and_bugs.ipynb#response-units) describes the change and its measured effect.
:::

## Design

Each draw comes from a known conditional distribution, so the exact 90%
interval is available as an oracle. Three generating processes are used:
homoscedastic normal noise, noise whose scale grows with `|x1|`, and
heavy-tailed t(3) noise. Each draw has 1000 training rows and 2000 test rows,
with 20 seeds per process. The full protocol is in
`interval_calibration/PROTOCOL.md`.

## Results

The table lists, for each process, the coverage gap from 0.90 with its
standard error across seeds and the mean interval width relative to the
oracle. The bold entry in each column is the best mboostrs or comparator value
for that column, excluding the oracle.

In [1]:
import json
from pathlib import Path

import pandas as pd
from great_tables import GT, loc, style

summary = json.loads(Path("interval_calibration/summary.json").read_text())
rows = []
for dgp, block in summary["dgps"].items():
    for method, s in block["methods"].items():
        rows.append({
            "process": dgp,
            "method": method,
            "gap": s["coverage_gap_pp"],
            "se": s["coverage_gap_se_pp"],
            "high": 100 * s["coverage_high_sd_tercile"],
            "width": s["width_vs_oracle_pct"],
        })
frame = pd.DataFrame(rows)

In [2]:
# Mark the smallest absolute coverage gap and the narrowest width per process,
# excluding the oracle row, so the reader can see the best method in each block.
best_width = frame[frame.method != "oracle"].groupby("process")["width"].idxmin()
best_gap = frame[frame.method != "oracle"].assign(absgap=lambda d: d.gap.abs()).groupby("process")["absgap"].idxmin()
frame["bold_width"] = frame.index.isin(best_width)
frame["bold_gap"] = frame.index.isin(best_gap)

table = (
    GT(frame.drop(columns=["bold_width", "bold_gap"]).round(2), rowname_col="method", groupname_col="process")
    .tab_header(
        title="Interval calibration at nominal 90% coverage",
        subtitle="20 seeds per process; test coverage gap in percentage points, width as percent over the oracle",
    )
    .tab_spanner(label="Coverage", columns=["gap", "se", "high"])
    .tab_spanner(label="Efficiency", columns=["width"])
    .cols_label(gap="Gap (pp)", se="SE (pp)", high="Top-scale tercile (%)", width="Width vs oracle (%)")
    .tab_footnote("Top-scale tercile: coverage among test rows whose true noise scale is in the highest third.", locations=loc.column_labels(columns="high"))
    .tab_footnote("SE: standard error of the gap across seeds.", locations=loc.column_labels(columns="se"))
    .tab_footnote("The oracle is the exact interval for the generating process, not a method.", locations=loc.row_groups())
)
for i in frame.index[frame.bold_gap]:
    table = table.tab_style(style.text(weight="bold"), locations=loc.body(columns="gap", rows=[list(frame.index).index(i)]))
for i in frame.index[frame.bold_width]:
    table = table.tab_style(style.text(weight="bold"), locations=loc.body(columns="width", rows=[list(frame.index).index(i)]))
table

GT(_tbl_data=            process          method   gap    se   high  width
0     homoscedastic    lightgbm_cqr  0.46  0.21  90.46  11.35
1     homoscedastic  lightgbm_split  0.26  0.31  90.26  12.35
2     homoscedastic    mboostrs_cqr  0.18  0.31  90.18   7.65
3     homoscedastic    mboostrs_cv+  0.28  0.26  90.28   7.39
4     homoscedastic  mboostrs_split  0.15  0.37  90.15   7.12
5     homoscedastic          oracle -0.01  0.12  89.98   0.00
6   heteroscedastic    lightgbm_cqr  0.27  0.23  87.02   8.81
7   heteroscedastic  lightgbm_split  0.35  0.32  78.64  23.84
8   heteroscedastic    mboostrs_cqr  0.27  0.25  89.71   6.73
9   heteroscedastic    mboostrs_cv+ -0.12  0.26  77.89  14.78
10  heteroscedastic  mboostrs_split -0.14  0.31  77.80  15.55
11  heteroscedastic          oracle -0.01  0.12  89.81   0.00
12  heavy_tailed_t3    lightgbm_cqr  0.47  0.26  90.48  18.63
13  heavy_tailed_t3  lightgbm_split  0.51  0.26  90.51  16.85
14  heavy_tailed_t3    mboostrs_cqr  0.41  0.21  90.42  16.67
15  heavy_tailed_t3    mboostrs_cv+  0.16  0.20  90.16   8.85
16  heavy_tailed_t3  mboostrs_split  0.22  0.20  90.22   9.63
17  heavy_tailed_t3          oracle  0.12  0.16  90.12   0.00, _body=<great_tables._gt_data.Body object at 0x10b9f8860>, _boxhead=Boxhead([ColInfo(var='process', type=<ColInfoTypeEnum.row_group: 3>, column_label='process', column_align='left', column_width=None), ColInfo(var='method', type=<ColInfoTypeEnum.stub: 2>, column_label='method', column_align='left', column_width=None), ColInfo(var='gap', type=<ColInfoTypeEnum.default: 1>, column_label='Gap (pp)', column_align='right', column_width=None), ColInfo(var='se', type=<ColInfoTypeEnum.default: 1>, column_label='SE (pp)', column_align='right', column_width=None), ColInfo(var='high', type=<ColInfoTypeEnum.default: 1>, column_label='Top-scale tercile (%)', column_align='right', column_width=None), ColInfo(var='width', type=<ColInfoTypeEnum.default: 1>, column_label='Width vs oracle (%)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x111131160>, _spanners=Spanners([SpannerInfo(spanner_id='Coverage', spanner_level=0, spanner_label='Coverage', spanner_units=None, spanner_pattern=None, vars=['gap', 'se', 'high'], built=None), SpannerInfo(spanner_id='Efficiency', spanner_level=0, spanner_label='Efficiency', spanner_units=None, spanner_pattern=None, vars=['width'], built=None)]), _heading=Heading(title='Interval calibration at nominal 90% coverage', subtitle='20 seeds per process; test coverage gap in percentage points, width as percent over the oracle', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11182b710>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11182b740>, _source_notes=[], _footnotes=[FootnoteInfo(locname=LocColumnLabels(columns='high'), grpname=None, colname='high', rownum=None, colnum=None, footnotes=['Top-scale tercile: coverage among test rows whose true noise scale is in the highest third.'], placement=<FootnotePlacement.auto: 3>), FootnoteInfo(locname=LocColumnLabels(columns='se'), grpname=None, colname='se', rownum=None, colnum=None, footnotes=['SE: standard error of the gap across seeds.'], placement=<FootnotePlacement.auto: 3>), FootnoteInfo(locname=LocRowGroups(rows=None), grpname='heteroscedastic', colname=None, rownum=None, colnum=None, footnotes=['The oracle is the exact interval for the generating process, not a method.'], placement=<FootnotePlacement.auto: 3>), FootnoteInfo(locname=LocRowGroups(rows=None), grpname='heavy_tailed_t3', colname=None, rownum=None, colnum=None, footnotes=['The oracle is the exact interval for the generating process, not a method.'], placement=<FootnotePlacement.auto: 3>), FootnoteInfo(locname=LocRowGroups(rows=None), grpname='homoscedastic', colname=None, rownum=None, colnum=None, footnotes=['The oracle is the exact interval for the generating process, not a method.'], placement=<FootnotePlacement.auto: 3>)], _

## Findings

The overall coverage is near 0.90 for every method on every process: all gaps
are within two standard errors of zero. The coverage gap does not separate
the methods, so this page scores calibration on the variance-adaptive
measures instead.

On the process with scale-dependent noise, the split conformal and CV+ intervals (intervals built from held-out residuals and from cross-validation residuals) have
constant width and cover only about 78% of rows in the highest-scale third.
The CQR interval (conformalized quantile regression: quantile fits for the lower and upper ends, widened by a conformal correction) widens with the noise and covers about 90% in that third.
This is the largest calibration gap in the study.

Interval width is 7 to 9% above the oracle on the homoscedastic and heavy-tailed
processes for the best mboostrs method, and CQR is 17% above the oracle on
heavy-tailed noise. That width gap is the efficiency loss from fitting and
calibrating on half the training rows; CQR's additional width on heavy tails
is a separate limitation of that method.

## Scorecard record

On the scorecard's 1-to-5 scale for D9 (uncertainty calibration), this study scores 3. Score 4 requires adaptive widening shown on
small-n and extrapolation cases too, and this study measures neither. The
machine-readable record is `interval_calibration/scorecard.json`. No other
dimension is measured here, and no aggregate is reported.